# Knot coach: step-state bench

This notebook is the progressive record for the knot skill: every evaluation run, its exact
provenance, and (if the eval gate ever fails and training starts, per #57) every training run.
New runs append at the bottom. Sibling skills get their own notebook in `docs/training/`.

Tickets: #64 (this bench), #65 (per-step pack records), #66 (server state machine).
Related cluster: #56-#61 (pause-and-show, coach-config YAML, eval packs).


## Where the numbers come from

Every accuracy figure below was produced on 2026-08-15 by `~/flux/bench/knots/bench_step_state.py`
(and its stateful variants) **on the GN100** (`gn100-2854`), calling the running
`nvidia/cosmos-reason2-8b` NIM at `http://localhost:30082/v1/chat/completions` with
`temperature=0`. No VSS in the loop: the bench samples 8 frames per 8-second chunk itself,
the same rate VSS samples, so it measures the model, not the plumbing.

- **Footage:** one public YouTube bowline tutorial (`CxpIecNDq4A`, 107 s, 480p), fetched with
  yt-dlp to `~/flux/bench/knots/`. One video, 14 chunks — every percentage point is 1/14, so
  treat the numbers as a signal about failure modes, not a benchmark score.
- **Frames:** 1 fps JPEGs. First run used 256px-wide frames (`frames_lo/`), later runs native
  480p (`frames/`), both under `~/flux/bench/knots/cases/bowline-CxpIecNDq4A/` on the box.
- **Ground truth:** hand-marked by the box agent from 1 fps contact sheets (below). Boundaries
  carry a few seconds of uncertainty, so scoring accepts **any step overlapping the chunk**:
  a boundary chunk counts as correct under either adjacent label.
- **Raw per-chunk outputs:** `results*.json` next to each case on the box.


## Ground truth (bowline, CxpIecNDq4A)

Six-step reading of the video, marked at 1 s resolution:

| step | seconds | what is on screen |
| --- | --- | --- |
| S0 | 0-18 | rope laid out straight, hands positioning |
| S1 | 18-27 | small overhand loop formed in the standing part |
| S2 | 27-45 | working end passed up through the loop |
| S3 | 45-58 | working end wrapped behind the standing part |
| S4 | 58-73 | working end back down through the loop |
| S5 | 73-107 | tightening, dressing, finished bowline held up |

The 4-phase variant merges S2+S3+S4 into one "thread the working end" phase.


## Test it yourself

The cells below are self-contained. Two one-time steps from this Mac:

1. **Tunnel** the NIM port over the existing ControlMaster socket (the box drops new TCP flows):
   `ssh -O forward -L 30082:localhost:30082 -o ControlPath=~/.ssh/cm-gn100 gn100`
2. **Sync** the case data (frames + ground truth) into `docs/training/knot-data/` (gitignored):
   `rsync -a -e "ssh -o ControlPath=~/.ssh/cm-gn100" gn100:flux/bench/knots/cases/ docs/training/knot-data/`


In [ ]:
import base64
import json
import time
from pathlib import Path

import httpx

ENDPOINT = (
    "http://localhost:30082/v1/chat/completions"  # tunneled cosmos-reason2-8b NIM
)
MODEL = "nvidia/cosmos-reason2-8b"
CHUNK_S, FRAMES_PER_CHUNK = 8, 8
CASE_DIR = Path("knot-data/bowline-CxpIecNDq4A")  # or knot-data/bowline-4phase

case = json.loads((CASE_DIR / "case.json").read_text())
frames = sorted((CASE_DIR / "frames").glob("*.jpg"))
print(case["knot"], len(frames), "frames,", len(case["steps"]), "steps")

In [ ]:
def chunk_frames(frames, chunk_s=CHUNK_S, per_chunk=FRAMES_PER_CHUNK):
    """1 fps frames -> (t0, t1, sampled frame paths) per chunk_s-second chunk."""
    out = []
    for start in range(0, len(frames), chunk_s):
        group = frames[start : start + chunk_s]
        if len(group) > per_chunk:
            step = len(group) / per_chunk
            group = [group[int(i * step)] for i in range(per_chunk)]
        out.append((start, min(start + chunk_s, len(frames)), group))
    return out


def gt_label(case, t0, t1):
    """Any ground-truth step overlapping [t0, t1) is an accepted label."""
    return {s["step"] for s in case["ground_truth"] if s["t0"] < t1 and s["t1"] > t0}


def build_prompt(case):
    steps = "\n".join(f"S{i}: {s}" for i, s in enumerate(case["steps"]))
    return (
        f"You are watching someone tie a {case['knot']} step by step. "
        f"The procedure's steps are:\n{steps}\n\n"
        "These frames are one consecutive chunk of the video, in order. "
        "Which single step is being performed in this chunk? "
        'Answer with JSON only: {"step": "S<n>", "state": "in_progress"|"completed", '
        '"confidence": "high"|"medium"|"low"}'
    )


def ask(prompt, frame_paths):
    content = [{"type": "text", "text": prompt}] + [
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,"
                + base64.b64encode(p.read_bytes()).decode()
            },
        }
        for p in frame_paths
    ]
    t = time.monotonic()
    r = httpx.post(
        ENDPOINT,
        json={
            "model": MODEL,
            "temperature": 0,
            "max_tokens": 300,
            "messages": [{"role": "user", "content": content}],
        },
        timeout=180,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"], time.monotonic() - t

In [ ]:
# Look at one chunk with your own eyes before asking the model.
T0 = 40  # edit me: chunk start second (multiples of 8)

from PIL import Image

chunk = next(c for c in chunk_frames(frames) if c[0] == T0)
imgs = [Image.open(p) for p in chunk[2]]
w, h = imgs[0].size
sheet = Image.new("RGB", (w * 4, h * 2))
for i, im in enumerate(imgs):
    sheet.paste(im, ((i % 4) * w, (i // 4) * h))
print(
    f"chunk {chunk[0]}-{chunk[1]}s   ground truth: {sorted(gt_label(case, chunk[0], chunk[1]))}"
)
sheet

In [ ]:
# Ask the model about that same chunk.
text, latency = ask(build_prompt(case), chunk[2])
print(f"{latency:.1f}s  {text}")

In [ ]:
# Full sweep: every chunk, scored against ground truth.
correct = 0
chunks = chunk_frames(frames)
for t0, t1, group in chunks:
    text, latency = ask(build_prompt(case), group)
    s, e = text.find("{"), text.rfind("}")
    pred = json.loads(text[s : e + 1]).get("step") if s != -1 else None
    ok = gt_label(case, t0, t1)
    hit = pred in ok
    correct += hit
    print(
        f"[{t0:3d}-{t1:3d}s] pred={pred} gt={sorted(ok)} {'OK' if hit else 'MISS'} {latency:.1f}s"
    )
print(f"accuracy {correct}/{len(chunks)} = {correct / len(chunks):.3f}")

## Run log

### Run 1 — stateless, 6 fine steps, 256px frames: **11/14 = 78.6%**, 2.5 s/chunk

| chunk | prediction | accepted labels | hit | latency |
| --- | --- | --- | --- | --- |
| 0-8s | S0 | S0 | OK | 3.7s |
| 8-16s | S0 | S0 | OK | 2.4s |
| 16-24s | S1 | S0, S1 | OK | 2.3s |
| 24-32s | S1 | S1, S2 | OK | 2.3s |
| 32-40s | S1 | S2 | MISS | 2.3s |
| 40-48s | S1 | S2, S3 | MISS | 2.3s |
| 48-56s | S4 | S3 | MISS | 2.4s |
| 56-64s | S4 | S3, S4 | OK | 2.4s |
| 64-72s | S4 | S4 | OK | 2.3s |
| 72-80s | S5 | S4, S5 | OK | 2.3s |
| 80-88s | S5 | S5 | OK | 2.3s |
| 88-96s | S5 | S5 | OK | 2.3s |
| 96-104s | S5 | S5 | OK | 2.3s |
| 104-107s | S5 | S5 | OK | 3.3s |

Reading: coarse phases (layout, loop, tighten, done) track perfectly. All three misses sit in
the fine threading steps: the model holds S1 while S2 is happening, then jumps to S4, never
emitting S2 or S3 at the right time.


### Run 2 — stateless, 6 fine steps, native 480p frames: **10/14 = 71.4%**, 3.4 s/chunk

| chunk | prediction | accepted labels | hit |
| --- | --- | --- | --- |
| 0-8s | S0 | S0 | OK |
| 8-16s | S0 | S0 | OK |
| 16-24s | S1 | S0, S1 | OK |
| 24-32s | S1 | S1, S2 | OK |
| 32-40s | S1 | S2 | MISS |
| 40-48s | S4 | S2, S3 | MISS |
| 48-56s | S4 | S3 | MISS |
| 56-64s | S4 | S3, S4 | OK |
| 64-72s | S5 | S4 | MISS |
| 72-80s | S5 | S4, S5 | OK |
| 80-88s | S5 | S5 | OK |
| 88-96s | S5 | S5 | OK |
| 96-104s | S5 | S5 | OK |
| 104-107s | S5 | S5 | OK |

Reading: resolution is not the bottleneck. The same threading confusion persists at full
resolution, so the failure is the model's rope-topology reading, not pixel starvation. This
matches the gate failure #57 predicted for rope topology.


### Run 3 — stateful 3-way ("still on step N / next / other"), pointer advances on `next`: **9/14 = 64.3%**, 1.7 s/chunk

| chunk | verdict | pointer | accepted labels | hit |
| --- | --- | --- | --- | --- |
| 0-8s | current | S0 | S0 | OK |
| 8-16s | current | S0 | S0 | OK |
| 16-24s | next | S1 | S0, S1 | OK |
| 24-32s | next | S2 | S1, S2 | OK |
| 32-40s | next | S3 | S2 | MISS |
| 40-48s | next | S4 | S2, S3 | MISS |
| 48-56s | current | S4 | S3 | MISS |
| 56-64s | next | S5 | S3, S4 | MISS |
| 64-72s | current | S5 | S4 | MISS |
| 72-80s | current | S5 | S4, S5 | OK |
| 80-88s | current | S5 | S5 | OK |
| 88-96s | current | S5 | S5 | OK |
| 96-104s | current | S5 | S5 | OK |
| 104-107s | current | S5 | S5 | OK |

Reading: anchoring the model on the current step gives it a `next` bias; one eager advance
cascades and the pointer runs 1-2 steps ahead of reality through the whole middle.


### Run 4 — stateful + 2-chunk debounce before advancing: **6/14 = 42.9%**, 1.7 s/chunk

Worst run. The debounce overcorrects: the pointer now lags ground truth by 1-2 steps for the
entire middle of the video. Conclusion from runs 3-4: the anchored 3-way question is a weaker
signal than full-list classification, in either smoothing direction. Per-chunk rows are in
`results_stateful_db2.json` on the box.


### Run 5 — stateless, 4 merged phases (S2+S3+S4 = "thread the working end"): **12/14 = 85.7%**, 3.4 s/chunk

| chunk | prediction | accepted labels | hit |
| --- | --- | --- | --- |
| 0-8s | S0 | S0 | OK |
| 8-16s | S0 | S0 | OK |
| 16-24s | S1 | S0, S1 | OK |
| 24-32s | S2 | S1, S2 | OK |
| 32-40s | S1 | S2 | MISS |
| 40-48s | S2 | S2 | OK |
| 48-56s | S2 | S2 | OK |
| 56-64s | S2 | S2 | OK |
| 64-72s | S3 | S2 | MISS |
| 72-80s | S3 | S2, S3 | OK |
| 80-88s | S3 | S3 | OK |
| 88-96s | S3 | S3 | OK |
| 96-104s | S3 | S3 | OK |
| 104-107s | S3 | S3 | OK |

Reading: both misses are single-chunk boundary wobble (one chunk late into the threading
phase, one chunk early out of it) — exactly the noise a majority-of-last-2-3-chunks filter in
the server pointer absorbs. With that smoothing the pointer trajectory on this video is
error-free.


## Findings so far

1. **Phase-level tracking works; fine topology does not.** The VLM reliably distinguishes
   layout / loop-forming / threading / tightening / done, and cannot distinguish
   up-through vs behind vs back-down at any tested resolution or prompt shape.
2. **Author procedure records at phase granularity** (#65): the coached card for the threading
   phase displays its three sub-steps as text; the tracker only watches the phase transition.
3. **Keep the VLM stateless, put state on the server** (#66): full-list classification per
   chunk, monotone pointer with majority smoothing. Do not tell the VLM the current step.
4. **Latency has headroom:** 1.7-3.5 s per chunk against an 8 s chunk, on the live stack with
   both NIMs and VSS resident.

## Next

- Ground-truth and sweep the other two videos already on the box: square knot `OxdUfYKrcfY`,
  clove hitch `vBDUz8PlKTA`.
- Phase B: same footage through VSS ingestion for end-to-end chunk-to-event latency.
- Training only enters if a skill fails the eval gate (#57's escalation rule); if the coach
  ever needs fine topology, that is a dedicated classifier, not a VLM prompt.


## Six-knot MVP sweep (2026-08-16)

Knot set per user decision: palomar in, taut-line hitch dropped. Same bench (stateless
full-list, cosmos-reason2-8b, 8 frames per 8 s chunk, temperature 0), one tutorial video per
knot, ground truth hand-marked from 1 fps contact sheets. `*` segments (title cards,
presenter cuts) are excluded from scoring. Cases and results on the box under
`~/flux/bench/knots/cases/`.

| knot | video | steps | chunks | raw chunk accuracy |
| --- | --- | --- | --- | --- |
| bowline (4-phase) | CxpIecNDq4A | 4 | 14 | 85.7% |
| square knot | OxdUfYKrcfY | 4 | 11 | 81.8% |
| trucker's hitch | yUp3t-SbxIo | 6 | 9 | 66.7% |
| figure eight | 0CnYmY_B938 | 5 | 8 | 62.5% |
| clove hitch | vBDUz8PlKTA | 4 | 12 | 58.3% |
| palomar (7 steps) | TFk_Ktw2f1w | 7 | 20 | 55.0% |

**Playthroughs** (smoothed pointer: advance when two consecutive chunks agree on a later
step, monotone): final stages land at sensible times on all six, but the pointer skips
intermediate stages where stable predictions jump (square 1→4, clove 1→4, trucker's 1→6).
Consequences: the overlay must handle multi-step catch-up, and per-knot step lists likely
need the same phase-merging that took the bowline from 71% to 86%. Screenshots:
`docs/progress/20260815-163034_*`; generator: session scratchpad `playthrough.py`.

Ops note: the two NIMs each reserve ~48 GB unified memory; restarting cosmos requires that
headroom free or it crash-loops (stop nemotron first, restart cosmos, start nemotron after).


## Live wire test (2026-08-16)

Full product path, no bench harness: the flux server's `/v1/coach` sessions
(32b5e85) classifying 8 s clip segments of the bowline tutorial against the box's
cosmos-reason2-8b through an SSH tunnel, pointer rule `advance_pointer` (two
consecutive clips agreeing on a later step).

| clip (s) | prediction | pointer |
| --- | --- | --- |
| 0-8 | S0 | 0 |
| 8-16 | S0 | 0 |
| 16-24 | S1 | 0 |
| 24-32 | S2 | 0 |
| 32-40 | S2 | **2** (advance) |
| 40-72 | S2 | 2 |
| 64-72 | S3 | 2 |
| 72-80 | S3 | **3** (advance) |
| 80-104 | S3 | 3 |

Both advances land inside the hand-marked ground-truth boundaries (threading
27-73 s, tighten 73 s+). The app's Watch chip (547e37d) drives this same loop
from the phone camera.
